In [19]:
import pandas as pd
import numpy as np

In [18]:
url = "https://raw.githubusercontent.com/EveliaCoss/CAMDA2025_metadatos/refs/heads/main/rawdata/TrainAndTest_cleaned/training_metadata_cleaned.tsv"
df = pd.read_csv(url, sep='\t')

In [2]:
df.to_csv("training_metadata_cleaned.csv", index=False)

In [ ]:
import pandas as pd
import numpy as np

def process_MIC_data_256(input_file, output_file):
    """
    Procesa un archivo CSV con datos MIC (Concentración Mínima Inhibitoria),
    recategoriza los valores MIC y asigna un fenotipo (Susceptible o Resistente)
    basado en tablas interpretativas específicas para cada género. Si no se puede
    recategorizar el MIC, se usa el fenotipo ya proporcionado en la columna 'phenotype'.
    Si 'new_genus' está vacío, se utiliza 'genus'.

    Además, los valores "Intermediate" se convierten en "Susceptible".

    Parámetros:
    -----------
    input_file : str
        Ruta al archivo CSV de entrada que contiene las columnas 'measurement_value', 
        'accession', 'new_genus', 'genus' y 'phenotype'.

    output_file : str
        Ruta del archivo CSV donde se guardará el DataFrame procesado.

    Retorna:
    --------
    df : pandas.DataFrame
        El DataFrame resultante con las columnas nuevas 'recategorized_MIC' y 'phenotype_assigned'.
    """

    df = pd.read_csv(input_file)
    df["measurement_value"] = pd.to_numeric(df["measurement_value"], errors="coerce")

    accessions_to_remove = [
        # Puedes reactivar esta lista si deseas excluir accesiones
    ]
    df = df[~df["accession"].isin(accessions_to_remove)]

    # Definir rangos y etiquetas para categorizar MIC
    bins = [0, 0.09, 0.185, 0.375, 0.75, 1.5, 3, 6, 12, 24, 48, 96, 192, 1200]
    labels = [0.06, 0.12, 0.25, 0.5, 1, 2, 4, 8, 16, 32, 64, 128, 256]
    recategorized_MIC = pd.cut(df["measurement_value"], bins=bins, labels=labels, right=False)
    df.insert(df.columns.get_loc("measurement_value") + 1, "recategorized_MIC", recategorized_MIC)

    phenotype_table = {
        "Klebsiella":              list('sssssssrrrrrr'),
        "Escherichia":             list('sssssssrrrrrr'),
        "Salmonella":              list('sssssssrrrrrr'),
        "Streptococcus":           list('ssssrrrrrrrrr'),
        "Staphylococcus":          list('sssssssrrrrrr'),
        "Pseudomonas":             list('sssssssssrrrr'),
        "Acinetobacter":           list('ssssssssrrrrr'),
        "Campylobacter":           list('ssssssssrrrrr'),
        "Neisseria":               list('sssssrrrrrrrr')
    }

    phenotype_df = pd.DataFrame(phenotype_table, index=labels).T

    def assign_phenotype(row):
        genus = row["new_genus"] if pd.notna(row["new_genus"]) else row.get("genus", np.nan)
        mic = row["recategorized_MIC"]
        
        if not pd.isna(mic) and genus in phenotype_df.index:
            return "Susceptible" if phenotype_df.loc[genus, mic] == "s" else "Resistant"
        elif pd.isna(mic) and not pd.isna(row.get("phenotype")):
            return row["phenotype"].capitalize()
        else:
            return np.nan

    df["phenotype_assigned"] = df.apply(assign_phenotype, axis=1)

    # Reemplazar 'Intermediate' por 'Susceptible'
    df["phenotype_assigned"] = df["phenotype_assigned"].replace("Intermediate", "Susceptible")

    # Insertar phenotype_assigned después de new_genus
    df.insert(df.columns.get_loc("new_genus") + 1, "phenotype_assigned", df.pop("phenotype_assigned"))

    df.to_csv(output_file, index=False)
    return df


In [ ]:
df_resultado = process_MIC_data_256(
    input_file="training_metadata_cleaned.csv",
    output_file="CAMDA25_training_con_MIC_y_fenotipo_256.csv"
)


In [ ]:
import pandas as pd
import numpy as np

def process_MIC_data_1024(input_file, output_file):
    """
    Procesa un archivo CSV con datos MIC (Concentración Mínima Inhibitoria),
    recategoriza los valores MIC y asigna un fenotipo (Susceptible o Resistente)
    basado en tablas interpretativas específicas para cada género. Si no se puede
    recategorizar el MIC, se usa el fenotipo ya proporcionado en la columna 'phenotype'.
    Si 'new_genus' está vacío, se utiliza 'genus'.

    Además, los valores "Intermediate" se convierten en "Susceptible".

    Parámetros:
    -----------
    input_file : str
        Ruta al archivo CSV de entrada.

    output_file : str
        Ruta del archivo CSV procesado.

    Retorna:
    --------
    df : pandas.DataFrame
        El DataFrame con 'recategorized_MIC' y 'phenotype_assigned'.
    """

    df = pd.read_csv(input_file)
    df["measurement_value"] = pd.to_numeric(df["measurement_value"], errors="coerce")

    accessions_to_remove = [
        # Puedes añadir accesiones aquí si es necesario excluirlas
    ]
    df = df[~df["accession"].isin(accessions_to_remove)]

    # MIC hasta 1024
    bins = [0, 0.09, 0.185, 0.375, 0.75, 1.5, 3, 6, 12, 24, 48, 96, 192, 384, 768, 2000]
    labels = [0.06, 0.12, 0.25, 0.5, 1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024]
    recategorized_MIC = pd.cut(df["measurement_value"], bins=bins, labels=labels, right=False)
    df.insert(df.columns.get_loc("measurement_value") + 1, "recategorized_MIC", recategorized_MIC)

    # Tabla extendida hasta 1024
    phenotype_table = {
        "Klebsiella":      list('sssssssrrrrrrrr'),
        "Escherichia":     list('sssssssrrrrrrrr'),
        "Salmonella":      list('sssssssrrrrrrrr'),
        "Streptococcus":   list('ssssrrrrrrrrrrr'),
        "Staphylococcus":  list('sssssssrrrrrrrr'),
        "Pseudomonas":     list('sssssssssrrrrrr'),
        "Acinetobacter":   list('ssssssssrrrrrrr'),
        "Campylobacter":   list('ssssssssrrrrrrr'),
        "Neisseria":       list('sssssrrrrrrrrrr')
    }

    phenotype_df = pd.DataFrame(phenotype_table, index=labels).T

    def assign_phenotype(row):
        genus = row["new_genus"] if pd.notna(row["new_genus"]) else row.get("genus", np.nan)
        mic = row["recategorized_MIC"]
        
        if not pd.isna(mic) and genus in phenotype_df.index:
            return "Susceptible" if phenotype_df.loc[genus, mic] == "s" else "Resistant"
        elif pd.isna(mic) and not pd.isna(row.get("phenotype")):
            return row["phenotype"].capitalize()
        else:
            return np.nan

    df["phenotype_assigned"] = df.apply(assign_phenotype, axis=1)

    # Reemplazar 'Intermediate' por 'Susceptible'
    df["phenotype_assigned"] = df["phenotype_assigned"].replace("Intermediate", "Susceptible")

    # Reubicar la columna phenotype_assigned después de new_genus
    df.insert(df.columns.get_loc("new_genus") + 1, "phenotype_assigned", df.pop("phenotype_assigned"))

    df.to_csv(output_file, index=False)
    return df


In [ ]:
df_resultado_1024 = process_MIC_data_1024(
    input_file="training_metadata_cleaned.csv",
    output_file="CAMDA25_training_con_MIC_y_fenotipo_1024.csv"
)